> Legacy/optional workflow. For author SLM-SQL + CSC baseline and CSC LoRA research, use **09 → 10 → 11** and `docs/CSC_LORA.md`. This notebook is not the canonical comparison against the author system.

# Phase 7: final evaluation

Run this notebook on Kaggle after the model and benchmark paths are set. It records three fixed seeds for BIRD, Spider when gold SQL is available, business questions, bootstrap 95% CIs, paired McNemar tests, robustness splits, and a deterministic error sample. Missing data or models are skipped with an explicit message.

In [ ]:
from pathlib import Path
import json
import os
import sys
from collections import defaultdict

_candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents, Path('/kaggle/working/SLM-PRJ')]
ROOT = next((p for p in _candidates if (p / 'pyproject.toml').is_file() and (p / 'benchmarks').is_dir()), Path('/kaggle/working/SLM-PRJ'))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
FINAL_DIR = ROOT / 'results' / 'final'
FINAL_DIR.mkdir(parents=True, exist_ok=True)
RUN_EVAL = False
MODEL_PATH = Path(os.environ['FINAL_MODEL_PATH']) if os.environ.get('FINAL_MODEL_PATH') else None
BIRD_DIR = Path(os.environ['BIRD_DATA_DIR']) if os.environ.get('BIRD_DATA_DIR') else Path('/kaggle/input/bird')
SPIDER_DIR = Path(os.environ['SPIDER_DATA_DIR']) if os.environ.get('SPIDER_DATA_DIR') else Path('/kaggle/input/spider')
SEEDS = [42, 43, 44]
print({'root': str(ROOT), 'model': str(MODEL_PATH), 'bird_exists': BIRD_DIR.exists(), 'spider_exists': SPIDER_DIR.exists()})

In [ ]:
from benchmarks.bird_loader import get_db_path as bird_db_path, load_bird_dev
from benchmarks.spider_loader import get_db_path as spider_db_path, load_spider_dev, load_spider_test
from business.create_business_db import create_database
from core.config import load_config
from evaluation.final_eval import evaluate_pipeline_records, load_json, mcnemar_test, write_json

def config_for(seed: int, mode: str) -> dict:
    config = load_config()
    if MODEL_PATH is None:
        raise FileNotFoundError('Set FINAL_MODEL_PATH before running model evaluation.')
    config['model']['backend'] = 'huggingface'
    config['model']['model_id'] = str(MODEL_PATH)
    config['model']['local_files_only'] = True
    config['model']['device'] = 'cuda'
    config['model']['dtype'] = 'float16'
    config['model']['quantization'] = 'none'
    config['seed'] = seed
    config['sampling']['mode'] = mode
    config['sampling']['fixed_k'] = 16
    config['sampling']['rounds'] = [4, 4, 8, 16]
    config['sampling']['max_candidates'] = 32
    return config

def run_split(records, db_path_fn, name: str, seed: int, mode: str = 'adaptive_sc'):
    path = FINAL_DIR / f'{name}_seed{seed}.json'
    report = evaluate_pipeline_records(records, db_path_fn, config_for(seed, mode), variant_name=name, output_path=path)
    payload = load_json(path)
    payload['provenance'] = {'dataset': name, 'seed': seed, 'mode': mode, 'db_root': str(ROOT)}
    write_json(path, payload)
    return payload

if RUN_EVAL and MODEL_PATH is not None and (BIRD_DIR / 'dev.json').exists():
    bird_records = load_bird_dev(BIRD_DIR)
    for seed in SEEDS:
        run_split(bird_records, lambda db_id: bird_db_path(BIRD_DIR, db_id), 'bird_dev_adaptive', seed, 'adaptive_sc')
        run_split(bird_records, lambda db_id: bird_db_path(BIRD_DIR, db_id), 'bird_dev_fixed16', seed, 'fixed_sc')
else:
    print('SKIP BIRD: set RUN_EVAL=True, FINAL_MODEL_PATH, and BIRD_DATA_DIR with dev.json.')

In [ ]:
if RUN_EVAL and MODEL_PATH is not None:
    for split, loader in [('dev', load_spider_dev), ('test', load_spider_test)]:
        split_file = SPIDER_DIR / f'{split}.json'
        if not split_file.exists():
            print(f'SKIP Spider {split}: {split_file} missing.')
            continue
        try:
            spider_records = loader(SPIDER_DIR)
        except ValueError as error:
            print(f'SKIP Spider {split}: {error}')
            continue
        for seed in SEEDS:
            run_split(spider_records, lambda db_id, root=SPIDER_DIR: spider_db_path(root, db_id), f'spider_{split}_adaptive', seed, 'adaptive_sc')
            run_split(spider_records, lambda db_id, root=SPIDER_DIR: spider_db_path(root, db_id), f'spider_{split}_fixed16', seed, 'fixed_sc')
else:
    print('SKIP Spider: RUN_EVAL=False or FINAL_MODEL_PATH is missing.')

business_questions = load_json(ROOT / 'business' / 'business_questions.json')
business_db = create_database(ROOT / 'business' / 'business.sqlite')
if RUN_EVAL and MODEL_PATH is not None:
    for seed in SEEDS:
        run_split(business_questions, lambda db_id: business_db, 'business_adaptive', seed, 'adaptive_sc')
        run_split(business_questions, lambda db_id: business_db, 'business_fixed16', seed, 'fixed_sc')
else:
    print('SKIP business model evaluation: RUN_EVAL=False or FINAL_MODEL_PATH is missing.')

In [ ]:
def report_flags(path: Path) -> tuple[list[str], list[bool]]:
    report = load_json(path)
    return [item['id'] for item in report.get('records', [])], [bool(item['correct']) for item in report.get('records', [])]

def paired_test(adaptive_path: Path, fixed_path: Path, output_name: str) -> dict:
    adaptive = load_json(adaptive_path).get('records', [])
    fixed = load_json(fixed_path).get('records', [])
    fixed_by_id = {item['id']: item for item in fixed}
    paired = [(item, fixed_by_id[item['id']]) for item in adaptive if item['id'] in fixed_by_id]
    test = mcnemar_test([a['correct'] for a, _ in paired], [f['correct'] for _, f in paired])
    result = {'adaptive_file': str(adaptive_path), 'fixed_file': str(fixed_path), 'paired_ids': [a['id'] for a, _ in paired], 'mcnemar': test}
    write_json(FINAL_DIR / output_name, result)
    return result

for stem in ['bird_dev', 'business']:
    adaptive = FINAL_DIR / f'{stem}_adaptive_seed42.json'
    fixed = FINAL_DIR / f'{stem}_fixed16_seed42.json'
    if adaptive.exists() and fixed.exists():
        print(stem, paired_test(adaptive, fixed, f'{stem}_mcnemar.json'))
    else:
        print(f'SKIP McNemar {stem}: paired reports are missing.')

In [ ]:
# Robustness uses only supplied datasets with gold SQL. Put child folders under ROBUSTNESS_DATA_DIR.
ROBUSTNESS_ROOT = Path(os.environ['ROBUSTNESS_DATA_DIR']) if os.environ.get('ROBUSTNESS_DATA_DIR') else None
if RUN_EVAL and MODEL_PATH is not None and ROBUSTNESS_ROOT and ROBUSTNESS_ROOT.exists():
    for child in sorted(ROBUSTNESS_ROOT.iterdir()):
        data_file = child / 'dev.json'
        if not data_file.exists():
            continue
        try:
            records = load_spider_dev(child)
            for seed in SEEDS:
                run_split(records, lambda db_id, root=child: spider_db_path(root, db_id), f'robust_{child.name}', seed, 'adaptive_sc')
        except (OSError, ValueError) as error:
            print(f'SKIP robustness {child.name}: {error}')
else:
    print('SKIP robustness: set ROBUSTNESS_DATA_DIR and RUN_EVAL=True when the splits are available.')

In [ ]:
# Keep a deterministic 100-example failure sample for manual taxonomy review.
error_rows = []
for report_path in sorted(FINAL_DIR.glob('*.json')):
    if 'mcnemar' in report_path.name:
        continue
    try:
        report = load_json(report_path)
    except json.JSONDecodeError:
        continue
    for row in report.get('records', []):
        if not row.get('correct'):
            error_rows.append({**row, 'report': report_path.name})
error_rows = sorted(error_rows, key=lambda row: (row['report'], row['id']))[:100]
write_json(FINAL_DIR / 'error_sample_100.json', {'n': len(error_rows), 'records': error_rows, 'selection': 'lexicographic report/id order'})
print({'error_sample': len(error_rows), 'output': str(FINAL_DIR / 'error_sample_100.json')})